In [2]:
print("Hello Structured Output!")

Hello Structured Output!


In [3]:
from dataclasses import dataclass
from openai import OpenAI
import os
import json
from pydantic import BaseModel
from typing import Literal

@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests accross all inference providers """

    name: str
    env_var: str
    is_true: bool
    base_url: str | None
    model: str


PROVIDERS = [
    Provider("Groq","GROQ_API_KEY",True,"https://api.groq.com/openai/v1","openai/gpt-oss-120b"),
    Provider("OpenRouter","OPENROUTER_API_KEY",True,"https://openrouter.ai/api/v1","stealth/space-bunny-alpha")
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")


def build_client(provider: Provider) -> OpenAI:

    api_key = os.getenv(provider.env_var)


    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )

def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")

Found a provider key.


In [ ]:
REVIEW_ANALYSIS_SYSTEM_PROMPT = """
You are a product-review sentiment analyst.

Read each review the user sends and classify its sentiment. We should also attach a confidence score for the sentiment
between 0 to 1 and a short explanation for the sentiment as a reason.

For each review, respond with ONLY a JSON object -  no other text or comments - in exactly the following shape:

{
    "sentiment": "positive" | "negative" | "neutral",
    "confidence": 0.0 to 1.0,
    "reason": "short explanation for the sentiment"
}

""".strip()

In [5]:
def review_analysis(prompt: str) -> str:

    provider = select_provider()
    client = build_client(provider)

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": REVIEW_ANALYSIS_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [12]:
class SentimentResult(BaseModel):
    """One contract for validating the llm response"""

    sentiment: Literal["positive","negative","neutral"]
    confidence: float
    reason: str

In [13]:
def validateResponse(raw_reply: SentimentResult) -> SentimentResult:
    try:
        parsed = json.loads(raw_reply)
        return SentimentResult(**parsed) # to check it is type of SentimentResult or not
    except json.JSONDecodeError:
        return f"Invalide JSON: {raw_reply}"

In [19]:
prompt ="""The customer service team was incredibly helpful and gave me a refund immediately, but the actual product arrived completely shattered and ruined my daughter's birthday surprise.
"""

raw_response = review_analysis(prompt)

validResponse = validateResponse(raw_response)

print(type(validResponse))
print(validResponse)

<class '__main__.SentimentResult'>
sentiment='negative' confidence=0.86 reason='The product arrived broken and ruined a birthday, which outweighs the positive note about helpful customer service.'
